# Speaker Linking Across Files with PyAnnote WeSpeaker

This notebook treats speaker identity linking as an embedding-and-clustering problem.

Core assumptions:
- The `speaker` field is only trusted within the same `source_audio`.
- Cross-file global speaker identity must come from speaker embeddings, not diarization labels.
- Reference-clip selection can use DNSMOS after clusters are built.

Model used here:
- `pyannote/wespeaker-voxceleb-resnet34-LM`


## What This Notebook Does

1. Load the raw combined manifest.
2. Keep only clean clips for speaker linking.
3. Extract one embedding per clip with PyAnnote WeSpeaker.
4. Visualize the embedding space.
5. Sweep clustering thresholds and inspect cluster quality.
6. Pick one reference clip per discovered speaker cluster using DNSMOS.


In [1]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchaudio
from pyannote.audio import Inference, Model
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

COMBINED_MANIFEST = Path('/mnt/nas05/data02/vincenzo/podcast_data/youtube/processed/manifest_combined_sliding.jsonl')
HF_TOKEN = os.environ.get('HF_TOKEN')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_NAME = 'pyannote/wespeaker-voxceleb-resnet34-LM'

if HF_TOKEN is None:
    print('HF_TOKEN is not set. Set it before running the embedding cells.')
print('DEVICE =', DEVICE)


/home2/vincenzo/stt4sg-transcribe/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home2/vincenzo/stt4sg-transcribe/.venv/lib/python3.11/site-packages/pyannote/audio/core/io.py:47: UserWarning: 
torchcodec is not installed correctly so built-in audio decoding will fail. Solutions are:
* use audio preloaded in-memory as a {'waveform': (channel, time) torch.Tensor, 'sample_rate': int} dictionary;
* fix torchcodec installation. Error message was:

Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6 and 7.
          2. The PyTorch version (2.8.0+cu128) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/p

HF_TOKEN is not set. Set it before running the embedding cells.
DEVICE = cpu


In [4]:
def load_jsonl(path: Path):
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return pd.DataFrame(rows)


df = load_jsonl(COMBINED_MANIFEST)
df  = df.head(100).reset_index(drop=True)
df['duration'] = df['end'] - df['start']
df['show'] = df['source_audio'].fillna('').map(lambda x: Path(x).stem)
df['speaker_local_key'] = df['source_audio'].fillna('') + '::' + df['speaker'].fillna('')
df['dnsmos_sig'] = pd.to_numeric(df.get('dnsmos_sig'), errors='coerce')
df['dnsmos_bak'] = pd.to_numeric(df.get('dnsmos_bak'), errors='coerce')
df['dnsmos_sum'] = df['dnsmos_sig'].fillna(0.0) + df['dnsmos_bak'].fillna(0.0)

clean = df[
    (df['audio_path'].notna())
    & (df['speaker'].notna())
    & (df['purity'].fillna(0.0) >= 0.98)
    & (df['coverage'].fillna(0.0) >= 0.95)
    & (df['duration'] >= 2.5)
    & (df['dnsmos_sig'].fillna(0.0) >= 3.0)
    & (df['dnsmos_bak'].fillna(0.0) >= 3.0)
].copy()

print('all rows:', len(df))
print('clean rows:', len(clean))
clean[['audio_path', 'source_audio', 'speaker', 'duration', 'purity', 'coverage', 'dnsmos_sig', 'dnsmos_bak']].head()


: 

In [ ]:
# Start within a single show first. Cross-show clustering is possible, but much harder.
top_show = clean['show'].value_counts().index[0]
work = clean[clean['show'] == top_show].copy()
print('selected show:', top_show)
print('rows in selected show:', len(work))
work[['audio_path', 'speaker', 'speaker_local_key', 'dnsmos_sum']].head()


In [ ]:
model = Model.from_pretrained(MODEL_NAME, use_auth_token=HF_TOKEN)
inference = Inference(model, window='whole')
inference.to(DEVICE)


def load_mono_16k(audio_path: str):
    wav, sr = torchaudio.load(audio_path)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != 16000:
        wav = torchaudio.functional.resample(wav, sr, 16000)
    return wav.squeeze(0).numpy(), 16000


def extract_embedding(audio_path: str):
    waveform, sample_rate = load_mono_16k(audio_path)
    emb = inference({'waveform': waveform[None, :], 'sample_rate': sample_rate})
    emb = np.asarray(emb).reshape(-1)
    emb = emb / np.linalg.norm(emb)
    return emb


In [ ]:
MAX_ROWS = 600
work = work.head(MAX_ROWS).copy()
work['embedding'] = work['audio_path'].map(extract_embedding)
X = np.stack(work['embedding'].to_list(), axis=0)
print('embedding matrix:', X.shape)


In [ ]:
pca = PCA(n_components=2, random_state=0)
xy = pca.fit_transform(X)
viz = work.copy()
viz['x'] = xy[:, 0]
viz['y'] = xy[:, 1]

plt.figure(figsize=(9, 7))
plt.scatter(viz['x'], viz['y'], s=18, alpha=0.65)
plt.title('PyAnnote WeSpeaker embeddings projected with PCA')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(alpha=0.2)
plt.show()


In [ ]:
thresholds = np.round(np.arange(0.15, 0.71, 0.025), 3)
rows = []

for thr in thresholds:
    clusterer = AgglomerativeClustering(
        n_clusters=None,
        metric='cosine',
        linkage='average',
        distance_threshold=float(thr),
    )
    labels = clusterer.fit_predict(X)
    n_clusters = int(pd.Series(labels).nunique())
    largest_cluster = int(pd.Series(labels).value_counts().iloc[0])
    sil = np.nan
    if n_clusters > 1:
        sil = float(silhouette_score(X, labels, metric='cosine'))
    rows.append(
        {
            'threshold': float(thr),
            'n_clusters': n_clusters,
            'largest_cluster': largest_cluster,
            'silhouette_cosine': sil,
        }
    )

sweep = pd.DataFrame(rows)
sweep.sort_values('silhouette_cosine', ascending=False).head(10)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(sweep['threshold'], sweep['n_clusters'], marker='o')
axes[0].set_title('Number of clusters vs threshold')
axes[0].set_xlabel('cosine distance threshold')
axes[0].set_ylabel('clusters')
axes[0].grid(alpha=0.2)

axes[1].plot(sweep['threshold'], sweep['silhouette_cosine'], marker='o')
axes[1].set_title('Silhouette vs threshold')
axes[1].set_xlabel('cosine distance threshold')
axes[1].set_ylabel('silhouette')
axes[1].grid(alpha=0.2)
plt.tight_layout()
plt.show()


In [ ]:
# Pick a threshold after looking at the sweep.
BEST_THRESHOLD = float(sweep.sort_values('silhouette_cosine', ascending=False).iloc[0]['threshold'])
print('BEST_THRESHOLD =', BEST_THRESHOLD)

clusterer = AgglomerativeClustering(
    n_clusters=None,
    metric='cosine',
    linkage='average',
    distance_threshold=BEST_THRESHOLD,
)
work['global_speaker_cluster'] = clusterer.fit_predict(X)

cluster_sizes = work['global_speaker_cluster'].value_counts().sort_index()
plt.figure(figsize=(10, 4))
plt.bar(cluster_sizes.index.astype(str), cluster_sizes.values)
plt.title('Cluster sizes')
plt.xlabel('cluster id')
plt.ylabel('rows')
plt.xticks(rotation=90)
plt.show()


In [ ]:
xy = PCA(n_components=2, random_state=0).fit_transform(X)
plot_df = work.copy()
plot_df['x'] = xy[:, 0]
plot_df['y'] = xy[:, 1]

plt.figure(figsize=(9, 7))
for cluster_id, chunk in plot_df.groupby('global_speaker_cluster'):
    plt.scatter(chunk['x'], chunk['y'], s=18, alpha=0.7, label=str(cluster_id))
plt.title('Embedding PCA colored by discovered cluster')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(alpha=0.2)
plt.legend(title='cluster', bbox_to_anchor=(1.02, 1), loc='upper left', ncol=1)
plt.show()


In [ ]:
# This checks a weak consistency signal: within one source_audio, local speaker labels should usually stay together.
consistency = (
    work.groupby(['source_audio', 'speaker', 'global_speaker_cluster'])
    .size()
    .rename('n')
    .reset_index()
    .sort_values(['source_audio', 'speaker', 'n'], ascending=[True, True, False])
)
consistency.head(30)


In [ ]:
# Reference clip selection per discovered speaker cluster.
# DNSMOS is a reasonable first pass for 'cleanest reference', especially with purity/coverage filters already applied.
reference_candidates = work.copy()
reference_candidates['reference_rank_score'] = (
    2.0 * reference_candidates['dnsmos_sig'].fillna(0.0)
    + 1.0 * reference_candidates['dnsmos_bak'].fillna(0.0)
    + 0.5 * reference_candidates['purity'].fillna(0.0)
    + 0.5 * reference_candidates['coverage'].fillna(0.0)
)

best_reference = (
    reference_candidates.sort_values(
        ['global_speaker_cluster', 'reference_rank_score', 'duration'],
        ascending=[True, False, False],
    )
    .groupby('global_speaker_cluster', as_index=False)
    .first()
)

best_reference[
    ['global_speaker_cluster', 'audio_path', 'source_audio', 'speaker', 'duration', 'dnsmos_sig', 'dnsmos_bak', 'purity', 'coverage']
].head(20)


## Practical Notes

- Start within one podcast/show. Global clustering across all shows can merge people who merely sound similar.
- Use conservative quality gates before embedding extraction.
- DNSMOS is a reasonable way to choose a reference clip after clustering, but not a full identity-confidence metric.
- After picking a threshold, listen to a few examples from each cluster before treating it as a real person-level identity.
- If a cluster contains multiple local speakers from the same `source_audio`, it is probably too loose.
